# ARA figures (ICSOC)
Fig 1, 3, 4a, 4b, 5. Run from the Firedragon folder.

In [ ]:
import os, json
from wandb.sdk.internal.datastore import DataStore
from wandb.proto import wandb_internal_pb2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator

plt.rcParams['pdf.fonttype'] = 42
os.makedirs('figures/notebook', exist_ok=True)

methods = ['RA', 'Reactive', 'Predictive', 'SARL']
colors = {'RA': '#0072B2', 'Reactive': '#E69F00', 'Predictive': '#009E73', 'SARL': '#CC79A7',
          'compute': '#56B4E9', 'network': '#D55E00', 'slo': 'grey'}

## Load the runs
The `.wandb` files of the Zenodo dataset, read with wandb's own reader.

In [ ]:
def load(path, method, events, bounds):
    ds = DataStore()
    ds.open_for_scan(path)
    rows = []
    while True:
        try:
            data = ds.scan_data()
        except Exception:  # run killed mid-write
            break
        if data is None:
            break
        rec = wandb_internal_pb2.Record()
        rec.ParseFromString(data)
        if rec.WhichOneof('record_type') == 'history':
            rows.append({'.'.join(i.nested_key) or i.key: json.loads(i.value_json) for i in rec.history.item})
    hist = pd.DataFrame(rows).dropna(subset=['total_latency.total_latency']).reset_index(drop=True)

    df = pd.DataFrame({'Method': method, 'Run': path.split('-')[-1][:-6], 't': hist['_runtime'],
                       'total': hist['total_latency.total_latency'],
                       'network': hist['total_latency.network_latency'],
                       'compute': hist['total_latency.compute_latency']})
    df['minute'] = df['t'] / 60
    df['phase'] = np.searchsorted(events, df['t'], side='right') + 1
    df[['nw_lo', 'nw_hi', 'cp_lo', 'cp_hi']] = [bounds[p - 1] for p in df['phase']]
    df['total_lo'] = df['nw_lo'] + df['cp_lo']
    df['total_hi'] = df['nw_hi'] + df['cp_hi']

    # skip: step 0, first step of a phase, 200 ms sentinel (locust csv cut), after the schedule
    sentinel = df['total'] == 200
    df.loc[sentinel, ['total', 'network']] = np.nan
    first = df['phase'] != df['phase'].shift()
    df['valid'] = (df.index > 0) & ~first & ~sentinel & (df['t'] <= 21700)

    df['viol_total'] = (df['total'] < df['total_lo']) | (df['total'] > df['total_hi'])
    df['viol_network'] = (df['network'] < df['nw_lo']) | (df['network'] > df['nw_hi'])
    df['viol_compute'] = (df['compute'] < df['cp_lo']) | (df['compute'] > df['cp_hi'])
    return df

In [ ]:
# 10 -> 5 users after 4340 s, SLO 1000-1500 ms per domain
exp1 = pd.concat([
    load('anna/dict/wandb/run-20260508_090016-qu1lewx3/run-qu1lewx3.wandb', 'RA', [4340], [(1000, 1500, 1000, 1500)] * 2),
    load('anna/dict/wandb/run-20260508_181637-x3vk2e7w/run-x3vk2e7w.wandb', 'Reactive', [4340], [(1000, 1500, 1000, 1500)] * 2),
    load('anna/dict/wandb/run-20260509_161951-apnyo5fi/run-apnyo5fi.wandb', 'Predictive', [4340], [(1000, 1500, 1000, 1500)] * 2),
    load('anna/dict/wandb/run-20260707_123518-hhbuo5ur/run-hhbuo5ur.wandb', 'SARL', [4340], [(1000, 1500, 1000, 1500)] * 2),
    load('anna/dict/wandb/run-20260707_172540-0ha1yqgh/run-0ha1yqgh.wandb', 'SARL', [4340], [(1000, 1500, 1000, 1500)] * 2),
], ignore_index=True)

# long runs, 30jun.sh: one phase every 4340 s, (nw_lo, nw_hi, cp_lo, cp_hi)
events_long = [4340, 8680, 13020, 17360]
slo_long = [(1000, 1500, 1000, 1500),  # P1 10 users
            (500, 1000, 500, 1000),    # P2 5 users
            (250, 750, 750, 1250),     # P3 tight network
            (750, 1250, 250, 750),     # P4 tight compute
            (1000, 1500, 1000, 1500)]  # P5 15 users
long = pd.concat([
    load('anna/dict/wandb/run-20260702_143317-84hiwelv/run-84hiwelv.wandb', 'RA', events_long, slo_long),
    load('anna/dict/wandb/run-20260703_023740-4xqj92zu/run-4xqj92zu.wandb', 'RA', events_long, slo_long),
    load('anna/dict/wandb/run-20260702_203529-vnwkitr9/run-vnwkitr9.wandb', 'Reactive', events_long, slo_long),
    load('anna/dict/wandb/run-20260703_083951-a2lfgmmi/run-a2lfgmmi.wandb', 'Reactive', events_long, slo_long),
    load('anna/dict/wandb/run-20260703_204414-jnzgyput/run-jnzgyput.wandb', 'Predictive', events_long, slo_long),
    load('anna/dict/wandb/run-20260704_084836-ijnqouk2/run-ijnqouk2.wandb', 'Predictive', events_long, slo_long),
    load('anna/dict/wandb/run-20260706_115701-l1j4bji5/run-l1j4bji5.wandb', 'SARL', events_long, slo_long),
    load('anna/dict/wandb/run-20260706_175912-5818x90j/run-5818x90j.wandb', 'SARL', events_long, slo_long),
], ignore_index=True)

## Fig 1 — reactive autoscaler (motivation)
Data reconstructed from the old figure.

In [ ]:
df = pd.read_csv('my-reconstuction/motivation_fig1_extracted.csv')
vir = plt.colormaps['viridis']
act_colors = {'decrease': vir(0.0), 'keep': vir(0.5), 'increase': '#56B4E9'}
act_markers = {'decrease': 'v', 'keep': '.', 'increase': '^'}

fig, ax = plt.subplots(figsize=(5.3, 2.3))
ax2 = ax.twinx()

ax.plot(df['step_min'], df['compute_response_ms'], ls='--', color='#E69F00', lw=1.5)
ax.axhline(1000, ls=':', color='0.6', lw=1)
ax.axhline(1500, ls=':', color='0.6', lw=1)
v = df[(df['compute_response_ms'] > 1500) | (df['compute_response_ms'] < 1000)]
ax.scatter(v['step_min'], v['compute_response_ms'], color='#C04657', marker='.', s=19, zorder=4)
ax.set_ylim(500, 2100)
ax.set_yticks([1000, 1500])
ax.set_yticklabels([1000, 1500], color='#E69F00', fontsize=13)
ax.set_ylabel('Response time (ms)', color='#E69F00', fontsize=14)
ax.tick_params(axis='x', labelbottom=False)

for i, a in enumerate(['decrease', 'keep', 'increase']):
    s = df[df['action'] == a]['step_min']
    ax2.scatter(s, [i] * len(s), marker=act_markers[a], s=40, color=act_colors[a])
ax2.set_ylim(-0.6, 2.6)
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['decrease', 'keep', 'increase'], fontsize=13, fontweight='bold')
for label in ax2.get_yticklabels():
    label.set_color(act_colors[label.get_text()])

plt.tight_layout()
plt.savefig('figures/notebook/fig_motivation.pdf', bbox_inches='tight')
plt.show()

## Fig 3 — Exp1 boxplot (10 → 5 users)

In [ ]:
df = exp1[exp1['valid']].copy()
df['label'] = df['phase'].map({1: '10 users', 2: '5 users'})

fig, ax = plt.subplots(figsize=(6, 5))
ax.axhspan(2000, 3000, color=colors['slo'], alpha=0.2, zorder=0)
sns.boxplot(data=df, x='label', y='total', hue='Method', hue_order=methods, palette=colors, saturation=1,
            boxprops=dict(alpha=0.6), linecolor='black', linewidth=2, ax=ax)
ax.set_ylim(0, 1.05 * max(df['total'].max(), df['total_hi'].max()))
ax.yaxis.set_major_locator(MultipleLocator(1000))
ax.grid(axis='y', color='grey', linestyle='--')
ax.set_xlabel('')
ax.set_ylabel('Resp. time (ms)', fontsize=28)
ax.tick_params(length=7, labelsize=24)
h, l = ax.get_legend_handles_labels()  # swap so rows read RA, Reactive / Predictive, SARL
ax.legend([h[0], h[2], h[1], h[3]], [l[0], l[2], l[1], l[3]], loc='lower center', bbox_to_anchor=(0.5, 1),
          ncol=2, frameon=False, fontsize=24, handlelength=1, handletextpad=0.4, columnspacing=0.8)

plt.tight_layout()
plt.savefig('figures/notebook/fig_wrapfig_exp1.pdf', bbox_inches='tight')
plt.show()

## Fig 4a — long runs, boxplot per phase

In [ ]:
df = long[long['valid']].copy()
labels = ['P1\n10u', 'P2\n5u', 'P3\nnw↓', 'P4\ncp↓', 'P5\n15u']
df['label'] = [labels[p - 1] for p in df['phase']]

fig, ax = plt.subplots(figsize=(8, 5))
for i, (lo, hi) in enumerate(df.groupby('phase')[['total_lo', 'total_hi']].first().values):
    ax.fill_between([i - 0.45, i + 0.45], lo, hi, color=colors['slo'], alpha=0.2, zorder=0)
sns.boxplot(data=df, x='label', y='total', hue='Method', order=labels, hue_order=methods, palette=colors,
            saturation=1, boxprops=dict(alpha=0.6), linecolor='black', linewidth=2, ax=ax)
ax.set_ylim(0, 1.05 * max(df['total'].max(), df['total_hi'].max()))
ax.yaxis.set_major_locator(MultipleLocator(1000))
ax.grid(axis='y', color='grey', linestyle='--')
ax.set_xlabel('')
ax.set_ylabel('Resp. time (ms)', fontsize=28)
ax.tick_params(length=7, labelsize=24)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncol=4, frameon=False, fontsize=24,
          handlelength=1, handletextpad=0.4, columnspacing=0.8)

plt.tight_layout()
plt.savefig('figures/notebook/fig_main_box_phases.pdf', bbox_inches='tight')
plt.show()

## Fig 4b — zoom on the P3 → P4 flip

In [ ]:
zoom_runs = {'RA': '84hiwelv', 'Reactive': 'vnwkitr9', 'Predictive': 'jnzgyput', 'SARL': '5818x90j'}
event = 13020 / 60
win = long[long['minute'].between(194, 246)]

fig, ax = plt.subplots(figsize=(8, 5))
ref = win[win['Run'] == '84hiwelv']
ax.fill_between(ref['minute'], ref['total_lo'], ref['total_hi'], step='mid', color=colors['slo'], alpha=0.2)
for m, run in zoom_runs.items():
    d = win[(win['Run'] == run) & win['valid']]
    ax.plot(d['minute'], d['total'], color=colors[m], linewidth=2, label=m)
    v = d[d['viol_total']]
    ax.scatter(v['minute'], v['total'], color=colors[m], edgecolors='black', linewidths=1, s=80, zorder=3)

top = (win[win['minute'].between(195, 245)]['total'].max() // 500 + 1) * 500
ax.set_xlim(195, 245)
ax.set_ylim(0, top)
ax.axvline(event, color='black', linestyle='--', linewidth=2)
ax.text(event - 1, 0.03 * top, 'P3', ha='right', va='bottom', color='grey', fontsize=26)
ax.text(event + 1, 0.03 * top, 'P4', ha='left', va='bottom', color='grey', fontsize=26)
ax.xaxis.set_major_locator(MultipleLocator(10))
ax.set_yticks([0, 1000, 2000, 3000])
ax.grid(True, alpha=0.3)
ax.set_xlabel('Time (min)', fontsize=28)
ax.set_ylabel('Resp. time (ms)', fontsize=28)
ax.tick_params(length=7, labelsize=24)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncol=4, frameon=False, fontsize=24,
          handlelength=1, handletextpad=0.4, columnspacing=0.8)

plt.tight_layout()
plt.savefig('figures/notebook/fig_main_ts_zoom.pdf', bbox_inches='tight')
plt.show()

## Fig 5 — per-domain view of one RA run

In [ ]:
d = long[long['Run'] == '84hiwelv']
ok = d[d['valid']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(d['minute'], d['cp_lo'], d['cp_hi'], step='mid', color=colors['compute'], alpha=0.2)
ax.fill_between(d['minute'], d['nw_lo'], d['nw_hi'], step='mid', color=colors['network'], alpha=0.2)
ax.fill_between(d['minute'], d['total_lo'], d['total_hi'], step='mid', color='grey', alpha=0.1)
ax.plot(ok['minute'], ok['compute'], '--', color=colors['compute'], linewidth=2, label='Compute')
ax.plot(ok['minute'], ok['network'], '--', color=colors['network'], linewidth=2, label='Network')
ax.plot(ok['minute'], ok['total'], color='grey', alpha=0.45, linewidth=2, label='Total')
v = ok[ok['viol_compute']]
ax.scatter(v['minute'], v['compute'], marker='.', s=60, color='#1A6699', zorder=3)
v = ok[ok['viol_network']]
ax.scatter(v['minute'], v['network'], marker='.', s=60, color='#8C3D00', zorder=3)

top = (ok['total'].max() // 500 + 1) * 500
end = d['minute'].max()
ax.set_xlim(0, end)
ax.set_ylim(0, top)
edges = [0] + [e / 60 for e in events_long] + [end]
for i in range(5):
    if i > 0:
        ax.axvline(edges[i], color='black', linestyle='--', linewidth=2)
    ax.text((edges[i] + edges[i + 1]) / 2, 0.97 * top, f'P{i + 1}', ha='center', va='top', color='grey', fontsize=26)
ax.xaxis.set_major_locator(MultipleLocator(60))
ax.grid(True, alpha=0.3)
ax.set_xlabel('Time (min)', fontsize=28)
ax.set_ylabel('Resp. time (ms)', fontsize=28)
ax.tick_params(length=7, labelsize=24)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncol=3, frameon=False, fontsize=24,
          handlelength=1, handletextpad=0.4, columnspacing=0.8)

plt.tight_layout()
plt.savefig('figures/notebook/fig_perdomain_ts.pdf', bbox_inches='tight')
plt.show()